# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.functions import regexp_replace, col, initcap, lower, trim, when
from pyspark.sql.types import DecimalType

# Criando o database silver

In [0]:
%sql
USE CATALOG projeto;
-- DROP DATABASE silver CASCADE; -- Executar se tiver uma silver já criada
CREATE DATABASE IF NOT EXISTS silver;

## Tabela: Atendentes

In [0]:
# CÉLULA: TRANSFORMAÇÃO - base_atendentes para Silver (CORRIGIDO)

# Lendo a tabela bronze
df_atendentes_bronze = spark.table("projeto.bronze.base_atendentes")
print(f"📊 Registros na bronze: {df_atendentes_bronze.count()}")

# Mostrar schema original
print("📋 SCHEMA ORIGINAL:")
df_atendentes_bronze.printSchema()

# TRANSFORMAÇÕES

df_atendentes_silver = (
    df_atendentes_bronze
    .dropDuplicates(["id_atendente"])
    .withColumn("nome_atendente", 
                F.initcap(F.trim(F.col("nome_atendente"))))
    .withColumn("nivel_atendimento", 
                F.col("nivel_atendimento").cast("int"))
    .withColumn("descricao_nivel",
                F.when(F.col("nivel_atendimento") == 1, "Nível 1 - Suporte Básico")
                 .when(F.col("nivel_atendimento") == 2, "Nível 2 - Suporte Especializado")
                 .when(F.col("nivel_atendimento") == 3, "Nível 3 - Suporte Avançado")
                 .otherwise("Nível Não Especificado"))
    .withColumn("id_atendente", F.col("id_atendente").cast("int"))
    .withColumn("nome_atendente", F.col("nome_atendente").cast("string"))
    .withColumn("nivel_atendimento", F.col("nivel_atendimento").cast("int"))
    .withColumn("descricao_nivel", F.col("descricao_nivel").cast("string"))
    .withColumn("silver_ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("source_file"))

    .select(
        "id_atendente",
        "nome_atendente", 
        "nivel_atendimento",
        "descricao_nivel",
        "silver_ingestion_timestamp",
        "source_file"
    )
)

df_atendentes_silver.printSchema()
display(df_atendentes_silver)

📊 Registros na bronze: 20
📋 SCHEMA ORIGINAL:
root
 |-- id_atendente: integer (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

root
 |-- id_atendente: integer (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: integer (nullable = true)
 |-- descricao_nivel: string (nullable = false)
 |-- silver_ingestion_timestamp: timestamp (nullable = false)
 |-- source_file: string (nullable = true)



id_atendente,nome_atendente,nivel_atendimento,descricao_nivel,silver_ingestion_timestamp,source_file
1,Ana Souza,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
2,Bruno Lima,2,Nível 2 - Suporte Especializado,2025-11-21T21:13:34.105Z,base_atendentes.csv
3,Carla Mendes,2,Nível 2 - Suporte Especializado,2025-11-21T21:13:34.105Z,base_atendentes.csv
4,Diego Rocha,2,Nível 2 - Suporte Especializado,2025-11-21T21:13:34.105Z,base_atendentes.csv
5,Elisa Santos,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
6,Felipe Araújo,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
7,Gabriela Nunes,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
8,Henrique Costa,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
9,Isabela Martins,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv
10,João Pedro,1,Nível 1 - Suporte Básico,2025-11-21T21:13:34.105Z,base_atendentes.csv


In [0]:
# VERIFICAÇÃO DOS VALORES PADRONIZADOS

print("VERIFICAÇÃO DOS VALORES PADRONIZADOS")

print("\nDISTRIBUIÇÃO POR NÍVEL DE ATENDIMENTO:")
distribuicao_nivel = df_atendentes_silver.groupBy("nivel_atendimento", "descricao_nivel").count().orderBy("nivel_atendimento")
display(distribuicao_nivel)

print("\nNOMES DE ATENDENTES (amostra):")
nomes_amostra = df_atendentes_silver.select("nome_atendente").collect()
for nome in nomes_amostra:
    print(f"  - {nome['nome_atendente']}")

print(f"\nTOTAL DE ATENDENTES POR NÍVEL:")
for row in distribuicao_nivel.collect():
    print(f"  - {row['descricao_nivel']}: {row['count']} atendentes")

print(f"\nATENDENTES DO NÍVEL 2 (Especializados):")
nivel_2 = df_atendentes_silver.filter(F.col("nivel_atendimento") == 2).select("id_atendente", "nome_atendente")
display(nivel_2)

VERIFICAÇÃO DOS VALORES PADRONIZADOS

DISTRIBUIÇÃO POR NÍVEL DE ATENDIMENTO:


nivel_atendimento,descricao_nivel,count
1,Nível 1 - Suporte Básico,14
2,Nível 2 - Suporte Especializado,6



NOMES DE ATENDENTES (amostra):
  - Ana Souza
  - Bruno Lima
  - Carla Mendes
  - Diego Rocha
  - Elisa Santos
  - Felipe Araújo
  - Gabriela Nunes
  - Henrique Costa
  - Isabela Martins
  - João Pedro
  - Karen Ribeiro
  - Lucas Almeida
  - Marina Ferraz
  - Nicolas Teixeira
  - Otávio Barros
  - Paula Carvalho
  - Rafael Moreira
  - Sabrina Queiroz
  - Thiago Pires
  - Vitória Gonçalves

TOTAL DE ATENDENTES POR NÍVEL:
  - Nível 1 - Suporte Básico: 14 atendentes
  - Nível 2 - Suporte Especializado: 6 atendentes

ATENDENTES DO NÍVEL 2 (Especializados):


id_atendente,nome_atendente
2,Bruno Lima
3,Carla Mendes
4,Diego Rocha
13,Marina Ferraz
14,Nicolas Teixeira
17,Rafael Moreira


In [0]:
# SALVANDO NA CAMADA SILVER

try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS projeto.silver")
except Exception as e:
    print(f"Erro ao criar schema: {e}")

# Salvar a tabela
try:
    (
        df_atendentes_silver
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("projeto.silver.dim_atendentes")
    )
    print("✅ TABELA SALVA COM SUCESSO!")
    print("📊 Tabela: projeto.silver.dim_atendentes")
    
except Exception as e:
    print(f"❌ Erro ao salvar tabela: {e}")

# Verificação final
try:
    df_verificacao = spark.table("projeto.silver.dim_atendentes")
    print(f"✅ Registros confirmados na silver: {df_verificacao.count()}")
    
    print("\n📋 SCHEMA DA TABELA SALVA:")
    df_verificacao.printSchema()
    
except Exception as e:
    print(f"❌ Erro na verificação: {e}")

✅ TABELA SALVA COM SUCESSO!
📊 Tabela: projeto.silver.dim_atendentes
✅ Registros confirmados na silver: 20

📋 SCHEMA DA TABELA SALVA:
root
 |-- id_atendente: integer (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: integer (nullable = true)
 |-- descricao_nivel: string (nullable = true)
 |-- silver_ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



## Tabela: Canais

Para a tabela de dm.canais, é preciso garantir que os tipos de dados bem como os nomes das colunas estejam corretos conforme o grupo definiu.

Schema final da tabela:

```
root
 |-- Nome_Canal: string (nullable = false)
 |-- Status_Canal: string (nullable = false)
```

In [0]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, upper, initcap, coalesce

# criação da sessão do Spark
spark = SparkSession.builder.appName("bronze_to_silver").getOrCreate()

In [0]:
# paths dos schemas
bronze_path = "workspace.bronze"
silver_path = "workspace.silver"

In [0]:
canais = spark.read.table(f"{bronze_path}.canais")

In [0]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

In [0]:
canais.display() # checando os dados

De acordo com o observado acima, a coluna ```nome_canal``` possui uma despadronização quanto à forma que as palavras estão escritas. Para padronizar, vamos capitalizar todas elas e caso tenha alguma ocorrência futura que seja nula, vamos colocar o valor de ```Desconhecido```.

Já na coluna de ```canal_status```, existe uma padronização quanto às diferentes escritas de inativo (sejam corretas no vocabulário português ou não). Para padronizar, vamos checar a primeira letra de cada ocorrência e atribuir à um valor constante, que será ```Ativo```, ```Inativo``` ou ```Desconhecido```, caso o valor da coluna seja nulo.

In [0]:
canais = (
    canais
    # nome_canal
    .withColumn("nome_canal", 
                # se o nome do canal for nulo, substitui por desconhecido
                coalesce(initcap(col("nome_canal")), lit("Desconhecido"))
    )
    .withColumnRenamed("nome_canal", "Nome_Canal")
    .withColumn("Nome_Canal", col("Nome_Canal").cast(StringType()))
    
    # canal_status
    .withColumn("canal_status", 
                when(upper(col("canal_status")).startswith("A"), "Ativo")
                .when(upper(col("canal_status")).startswith("I"), "Inativo")
                .otherwise("Desconhecido")
    )
    .withColumnRenamed("canal_status", "Status_Canal")
)

In [0]:
# checando alterações pós tratamento
canais.display()

In [0]:
# checando o schema atual dos dados da bronze layer
canais.printSchema()

Todas as mudanças foram efetivas e deixou a coluna padronizada para o futuro.

Como existem somente 6 ocorrências dos dados, não é possível criar futuras Views somente com esta tabela, somente em conjunto de outras tabelas.

Com isso, resta partir para o salvamento da tabela na Silver Layer.

In [0]:
canais.write.format("delta").mode("overwrite").saveAsTable(f"{silver_path}.dim_canais")

## Tabela: Chamados

## Tabela: Chamados_Hora

In [0]:
# Lendo a tabela chamados_hora da camada bronze e visualizando os primeiros registros
df_bz = spark.table("bronze.chamados_hora")
print(f"bronze.chamados_hora: {df_bz.count()} rows")
display(df_bz.limit(10))

In [0]:
# Remoção de duplicatas:
df_max = (
    df_bz.groupBy("ID_Chamado")
         .agg(F.max("ingestion_timestamp").alias("ingestion_timestamp"))
)
df_bz = df_bz.join(df_max, on=["ID_Chamado", "ingestion_timestamp"], how="inner")


# Renomeando as colunas para snake_case
df = (
    df_bz
    .withColumnRenamed("ID_Chamado", "id_chamado")
    .withColumnRenamed("ID_Cliente", "id_cliente")
    .withColumnRenamed("Hora_Abertura_Chamado", "hora_abertura_chamado_raw")
    .withColumnRenamed("Hora_Inicio_Atendimento", "hora_inicio_atendimento_raw")
    .withColumnRenamed("Hora_Finalizacao_Atendimento", "hora_finalizacao_atendimento_raw")
    # ingestion_timestamp já está em snake_case
)

# Criando uma função para tirar o " �s " e converter pra timestamp
def to_ts(col):
    return F.to_timestamp(F.regexp_replace(col, " �s ", " "), "dd/MM/yyyy HH:mm:ss")

df = (
    # Aplicando a função nas três colunas de tempo e dropando as raws
    df
    .withColumn("data_hora_abertura", to_ts(F.col("hora_abertura_chamado_raw")))
    .withColumn("data_hora_inicio_atendimento", to_ts(F.col("hora_inicio_atendimento_raw")))
    .withColumn("data_hora_finalizacao_atendimento", to_ts(F.col("hora_finalizacao_atendimento_raw")))
    .drop(
        "hora_abertura_chamado_raw",
        "hora_inicio_atendimento_raw",
        "hora_finalizacao_atendimento_raw"
    )

    # garantir tipos dos IDs em long
    .withColumn("id_chamado", F.col("id_chamado").cast("long"))
    .withColumn("id_cliente", F.col("id_cliente").cast("long"))

    # Criando duas métricas úteis de minutos
    .withColumn(
        "tempo_espera_atendimento_min",
        F.round((F.col("data_hora_inicio_atendimento").cast("long") - F.col("data_hora_abertura").cast("long")) / 60.0, 2)
    )

    .withColumn(
        "tempo_atendimento_min",
        F.round(
            (F.col("data_hora_finalizacao_atendimento").cast("long") - F.col("data_hora_inicio_atendimento").cast("long")) / 60.0, 2)
    )
)

# Ordenando as colunas
df = df.select(
    "id_chamado",
    "id_cliente",
    "data_hora_abertura",
    "data_hora_inicio_atendimento",
    "data_hora_finalizacao_atendimento",
    "tempo_espera_atendimento_min",
    "tempo_atendimento_min",
    "ingestion_timestamp"
)

display(df.limit(10))


In [0]:
# Salvando no silver.dim_chamado_hora (formato delta por padrão)
df.write.mode("overwrite").saveAsTable("silver.dim_chamado_hora")
print(f"silver.dim_chamado_hora: {df.count()} rows")

## Tabela: Clientes

## Tabela: Custos

In [0]:
# Lendo a tabela e vizualizando os primeiros registros
df_custos_bronze = spark.table("bronze.custos")
print(f"{df_custos_bronze.count()} rows")
display(df_custos_bronze.limit(10))

In [0]:
# Removendo duplicatas
df_custos_bronze = df_custos_bronze.distinct()

# Limpando os dados, manténdo apenas números, vírgula e ponto
df_clean = df_custos_bronze.withColumn(
    "custo_limpo",
    regexp_replace(col("custo"), "[^0-9,\\.]", "") 
)

# Trocando vírgula por pontos 
df_clean = df_clean.withColumn(
    "custo_padronizado",
    regexp_replace(col("custo_limpo"), ",", ".")
)


# Alterando o tipo de dados de custo para decimal
df_clean = df_clean.withColumn(
    "custo_final",
    col("custo_padronizado").cast(DecimalType(18, 10))
)

# Padronizando nome das colunas com snake_case
df_clean = df_clean.withColumnRenamed("custo_final", "valor_custo")

df_custos_silver = df_clean.select(
    "id_custo",
    "id_chamado",
    "valor_custo"
)

# Atualizando tempo de ingenstão para camanda silver
df_custos_silver = df_custos_silver.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
df_custos_silver.write.mode("overwrite").saveAsTable("silver.ft_custos")

In [0]:
print(f"{df_custos_silver.count()} rows")
display(df_custos_silver.limit(10))

## Tabela: Motivos

In [0]:
# Lendo e vizualizando dataset 
df_motivos_bronze = spark.table("bronze.base_motivos")
print(f"{df_motivos_bronze.count()} rows")
display(df_motivos_bronze.limit(10))


In [0]:
# Removendo duplicadas 
df_motivos_bronze = df_motivos_bronze.distinct()

# Removendo acentos
df_motivos_bronze = (
    df_motivos_bronze
    .withColumn("criticidade", regexp_replace("criticidade", "[áàâãä]", "a"))
    .withColumn("criticidade", regexp_replace("criticidade", "[éèêë]", "e"))
    .withColumn("criticidade", regexp_replace("criticidade", "[íìîï]", "i"))
    .withColumn("criticidade", regexp_replace("criticidade", "[óòôõö]", "o"))
    .withColumn("criticidade", regexp_replace("criticidade", "[úùûü]", "u"))
    .withColumn("criticidade", regexp_replace("criticidade", "ç", "c"))
)

# Se baseando nos crietrios de categorização e nos motivos mais comuns entre as demandas de atendimento essa perta atribui as categorias de cada motivo atraves de Keywords presentes no seu nome do motivo
df_motivos_bronze = df_motivos_bronze.withColumn(
            "categoria",
            when(col("nome_motivo").rlike("(?i)fatura|limite|contrato|dívida|renegociação"), "Financeiro")
            .when(col("nome_motivo").rlike("(?i)cart[aã]o|compra"), "Cartão")
            .when(col("nome_motivo").rlike("(?i)dados|telefone|email|agência|vencimento|cancel|encerr|fechamento"), "Cadastral")
            .when(col("nome_motivo").rlike("(?i)aplicativo|app|site|chatbot|ura"), "Atendimento")
            .when(col("nome_motivo").rlike("(?i)pontos|benef"), "Benefícios")
            .otherwise(None)
        )


# Padronizando formato de escrita coluna de criticidade
df_motivos_bronze = df_motivos_bronze.withColumn("criticidade", initcap(lower(trim(col("criticidade")))))

# Removendo espaços extra no nome
df_motivos_bronze = df_motivos_bronze.withColumn("nome_motivo", trim(col("nome_motivo")))

df_motivos_silver = df_motivos_bronze.withColumn("ingestion_timestamp", current_timestamp())


In [0]:
df_motivos_silver.write.mode("overwrite").saveAsTable("silver.dim_motivos")

In [0]:
print(f"{df_motivos_silver.count()} rows")
display(df_motivos_silver)

## Tabela: Pesquisa_Satisfação

In [0]:
from pyspark.sql import functions as F

df_bronze_satisfacao = spark.table("projeto.bronze.pesquisa_satisfacao") 

total_linhas = df_bronze_satisfacao.count()
total_chamados_unicos = df_bronze_satisfacao.select("id_chamado").distinct().count()
total_pesquisas_unicas = df_bronze_satisfacao.select("id_pesquisa").distinct().count()
qtd_duplicatas_chamado = total_linhas - total_chamados_unicos

qtd_nulos_nota = df_bronze_satisfacao.filter(F.col("nota_atendimento").isNull()).count()

df_bronze_satisfacao_outliers = df_bronze_satisfacao.filter(
    (F.col("nota_atendimento") < 1) | 
    (F.col("nota_atendimento") > 5)
)
qtd_outliers = df_bronze_satisfacao_outliers.count()
df_bronze_satisfacao_dados_incompletos = df_bronze_satisfacao.filter(
    F.col("id_chamado").isNull() |
    F.col("id_pesquisa").isNull() |
    F.col("nota_atendimento").isNull() |
    F.col("ingestion_timestamp").isNull()
)

print("RESUMO DE QUALIDADE DE DADOS")
print(f"Total de Linhas:{total_linhas}")
print(f"Chamados Únicos:{total_chamados_unicos}")
print(f"IDs Pesquisa Únicos:{total_pesquisas_unicas}")
print(f"Duplicatas de Chamado:{qtd_duplicatas_chamado}")
print(f"Notas Nulas:{qtd_nulos_nota}")
print(f"Notas Outliers (<1 ou >5):{qtd_outliers}")
print(f"Registros Incompleto{df_bronze_satisfacao_dados_incompletos.count()}")

if df_bronze_satisfacao_dados_incompletos.count() > 0:
    print("Visualizando amostra de dados incompletos:")
    display(df_bronze_satisfacao_dados_incompletos.limit(5))

if qtd_outliers > 0:
    print("Visualizando amostra de outliers:")
    display(df_bronze_satisfacao_outliers.limit(5))
     



PERCEBO QUE EXISTEM MUITOS VALORES DE NOTA_ATENDIMENTO FALTANTE (FAZ SENTIDO POIS OS CLIENTES NÃO SÃO OBRIGADOS A ATRIBUIR UMA NOTA AO FUNCIONÁRO), MAS OS CAMPOS DE ID ESTÃO SEMPRE COMO ÚNICOS POR ENQUANTO, MAS PRECISO GARANTIR NO MEU JOB QUE ESSAS LINHAS NÃO TERÃO DUPLICATAS

AGORA IREI FAZER UM CAST NOS MEUS DADOS


In [0]:
df_bronze_satisfacao = (
    df_bronze_satisfacao
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    .withColumn("id_pesquisa", F.col("id_pesquisa").cast("int"))
    .withColumn("nota_atendimento", F.col("nota_atendimento").cast("int"))
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
)

In [0]:
df_bronze_satisfacao.write.mode("overwrite").saveAsTable("projeto.silver.pesquisa_satisfacao")